# Accountant Agent Improvement Notebook

This notebook is a working playbook for analyzing and improving the `AccountantAgent` runtime.

It includes:
- A visual agent graph (runtime and tool dependencies)
- Prompt and tool-surface introspection
- A lightweight quality-evaluation harness
- A prioritized improvement backlog and rollout workflow

Run cells top-to-bottom.

In [3]:
from __future__ import annotations

import ast
import json
import subprocess
import textwrap
from pathlib import Path
from typing import Any

try:
    from IPython.display import Markdown, display
except Exception:  # pragma: no cover - notebook fallback
    Markdown = None
    display = print

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
AGENT_RUNTIME_PATH = ROOT / "services" / "agent" / "app" / "runtime" / "accountant.py"
AGENT_DOC_PATH = ROOT / "docs" / "agents" / "accountant.md"

print(f"Repo root: {ROOT}")
print(f"Runtime file exists: {AGENT_RUNTIME_PATH.exists()}")
print(f"Docs file exists: {AGENT_DOC_PATH.exists()}")

Repo root: /Users/valentinbarakov/proecti/agents
Runtime file exists: True
Docs file exists: True


## 1) Agent Graph (Architecture)

Use this graph to reason about where improvements should land.

```mermaid
graph TD
    Registry["runtime/registry.py"] --> Accountant["AccountantAgent"]
    Accountant --> BaseAgent["BaseAgent"]
    Accountant --> Prompt["get_system_prompt()"]
    Accountant --> Tools["get_tools()"]

    Tools --> Rag["rag_search"]
    Tools --> Calc["calculator"]
    Tools --> Dates["date_helper"]
    Tools --> Financial["financial tools"]
    Tools --> Partners["partner tools"]
    Tools --> CompanyBook["companybook tools"]

    Rag --> KnowledgeBase["Knowledge Base /retrieve"]
    Financial --> BusinessService["Business Service"]
    Partners --> BusinessService
    CompanyBook --> CompanyBookBG["CompanyBook.BG"]
    CompanyBook --> BusinessService

    Accountant --> Scope{"agent.company_id ?"}
    Scope -->|yes| SingleCompany["strict single-company scope"]
    Scope -->|no| ResolveCompany["list/resolve company first"]
```

### Improvement Lens

For each node above, ask:
1. Is behavior deterministic and test-covered?
2. Are failures explicit and user-recoverable?
3. Is the prompt/tool contract unambiguous?

In [4]:
def load_source(path: Path) -> str:
    if not path.exists():
        raise FileNotFoundError(path)
    return path.read_text(encoding="utf-8")


def extract_accountant_runtime_facts(source: str) -> dict[str, Any]:
    tree = ast.parse(source)
    facts: dict[str, Any] = {
        "imports": [],
        "tool_builders": [],
        "conditional_company_tools": False,
    }

    for node in ast.walk(tree):
        if isinstance(node, ast.ImportFrom) and node.module and node.module.startswith("app.tools"):
            names = [alias.name for alias in node.names]
            facts["imports"].append({"module": node.module, "names": names})

        if isinstance(node, ast.Call) and isinstance(node.func, ast.Name):
            if node.func.id.startswith("build_"):
                facts["tool_builders"].append(node.func.id)

        if isinstance(node, ast.If):
            text = ast.unparse(node.test) if hasattr(ast, "unparse") else ""
            if "company_id is None" in text:
                facts["conditional_company_tools"] = True

    # normalize
    facts["tool_builders"] = sorted(set(facts["tool_builders"]))
    return facts


runtime_source = load_source(AGENT_RUNTIME_PATH)
runtime_facts = extract_accountant_runtime_facts(runtime_source)
runtime_facts

{'imports': [{'module': 'app.tools.calculator', 'names': ['calculator']},
  {'module': 'app.tools.companybook', 'names': ['build_companybook_tools']},
  {'module': 'app.tools.dates', 'names': ['date_tool']},
  {'module': 'app.tools.financial',
   'names': ['build_company_tools',
    'build_financial_tools',
    'build_partner_tools']},
  {'module': 'app.tools.rag', 'names': ['build_rag_search_tool']}],
 'tool_builders': ['build_company_tools',
  'build_companybook_tools',
  'build_financial_tools',
  'build_partner_tools',
  'build_rag_search_tool'],
 'conditional_company_tools': True}

In [5]:
PROMPT_RULE_KEYWORDS = [
    "rag_search at most once",
    "create_invoice",
    "confirmed=false",
    "record_expense",
    "date_helper",
    "totals_by_currency",
    "BGN",
    "company_id",
]


def scan_keywords(source: str, keywords: list[str]) -> dict[str, bool]:
    normalized = source.lower()
    return {kw: kw.lower() in normalized for kw in keywords}


keyword_hits = scan_keywords(runtime_source, PROMPT_RULE_KEYWORDS)
print("Prompt/logic keyword coverage:")
for k, ok in keyword_hits.items():
    print(f"- {k}: {'YES' if ok else 'NO'}")


if Markdown and display:
    missing = [k for k, ok in keyword_hits.items() if not ok]
    if missing:
        display(Markdown("### Missing guardrail keywords\n" + "\n".join(f"- `{m}`" for m in missing)))
    else:
        display(Markdown("### Guardrail keyword check\nAll expected keywords were detected in runtime source."))

Prompt/logic keyword coverage:
- rag_search at most once: YES
- create_invoice: YES
- confirmed=false: YES
- record_expense: YES
- date_helper: YES
- totals_by_currency: YES
- BGN: YES
- company_id: YES


### Guardrail keyword check
All expected keywords were detected in runtime source.

## 2) Evaluation Harness (Behavior Quality)

This section gives you a repeatable way to score outputs after prompt/tool changes.

### Core dimensions
- Correctness (factually and financially correct)
- Policy compliance (confirmation rules, company scope, currency rule)
- Tool discipline (no unnecessary repeated retrieval/tool loops)
- Recovery quality (clear next steps when data is missing or conflicting)
- UX quality (concise, actionable, low ambiguity)

In [6]:
EVAL_CASES: list[dict[str, Any]] = [
    {
        "id": "tax-rag-single-call",
        "user_prompt": "How is VAT handled for this quarter for my company?",
        "expected": [
            "Uses retrieved tax/company-doc context",
            "Does not spam repeated retrieval calls for same question",
            "States assumptions clearly",
        ],
        "must_not": ["Invent regulation article", "No-source legal certainty claim"],
    },
    {
        "id": "invoice-create-two-step",
        "user_prompt": "Create an invoice to Acme for 1200 EUR consulting.",
        "expected": [
            "Draft first with confirmation request",
            "No persistence before explicit confirmation",
            "Asks only for missing required fields",
        ],
        "must_not": ["Immediate final creation without confirmation"],
    },
    {
        "id": "expense-confirmation",
        "user_prompt": "Record office rent expense for April: 2000 BGN.",
        "expected": [
            "Summarizes expense before write",
            "Requests explicit confirmation",
        ],
        "must_not": ["Persists expense before confirmation"],
    },
    {
        "id": "currency-threshold-logic",
        "user_prompt": "Are we above Bulgarian VAT threshold with these EUR invoices?",
        "expected": [
            "Explains conversion to BGN",
            "Separates source-currency reporting from threshold comparison",
        ],
        "must_not": ["Compares EUR totals directly to BGN threshold"],
    },
]

RUBRIC = {
    "correctness": 0.30,
    "policy_compliance": 0.30,
    "tool_discipline": 0.15,
    "recovery_quality": 0.15,
    "ux_quality": 0.10,
}


def weighted_score(scores: dict[str, float], weights: dict[str, float]) -> float:
    return round(sum(scores[k] * weights[k] for k in weights), 3)


# Example manual scoring template for one run
sample_scores = {
    "correctness": 0.9,
    "policy_compliance": 1.0,
    "tool_discipline": 0.8,
    "recovery_quality": 0.8,
    "ux_quality": 0.9,
}

print("Weighted score example:", weighted_score(sample_scores, RUBRIC))
print("\nEval cases:")
print(json.dumps(EVAL_CASES, indent=2, ensure_ascii=True))

Weighted score example: 0.9

Eval cases:
[
  {
    "id": "tax-rag-single-call",
    "user_prompt": "How is VAT handled for this quarter for my company?",
    "expected": [
      "Uses retrieved tax/company-doc context",
      "Does not spam repeated retrieval calls for same question",
      "States assumptions clearly"
    ],
    "must_not": [
      "Invent regulation article",
      "No-source legal certainty claim"
    ]
  },
  {
    "id": "invoice-create-two-step",
    "user_prompt": "Create an invoice to Acme for 1200 EUR consulting.",
    "expected": [
      "Draft first with confirmation request",
      "No persistence before explicit confirmation",
      "Asks only for missing required fields"
    ],
    "must_not": [
      "Immediate final creation without confirmation"
    ]
  },
  {
    "id": "expense-confirmation",
    "user_prompt": "Record office rent expense for April: 2000 BGN.",
    "expected": [
      "Summarizes expense before write",
      "Requests explicit confirma

## 3) Improvement Backlog (Prioritized)

### P0 (Safety / correctness)
- Tighten tool-level invariants around `confirmed` write operations (invoice/expense)
- Add explicit test assertions for company scope enforcement (`company_id` assigned vs unassigned)
- Add regression tests for currency-threshold comparisons (BGN conversion path)

### P1 (Quality / reliability)
- Add response templates for missing field requests to reduce ambiguity
- Add retry/backoff + typed error mapping for transient downstream tool failures
- Improve RAG deduplication heuristics for near-identical retrieval calls

### P2 (Productivity / observability)
- Add per-tool latency and error-rate dashboard slices
- Track "clarification question count" as a proxy for prompt clarity
- Add golden conversation snapshots for accountant critical workflows

In [7]:
IMPROVEMENT_TRACKER: list[dict[str, str]] = [
    {"id": "P0-1", "task": "Confirmation write invariants", "owner": "", "status": "todo"},
    {"id": "P0-2", "task": "Company scope regression tests", "owner": "", "status": "todo"},
    {"id": "P0-3", "task": "BGN threshold test suite", "owner": "", "status": "todo"},
    {"id": "P1-1", "task": "Missing-fields response templates", "owner": "", "status": "todo"},
    {"id": "P1-2", "task": "Downstream error mapping", "owner": "", "status": "todo"},
    {"id": "P2-1", "task": "Tool latency + error observability", "owner": "", "status": "todo"},
]

for row in IMPROVEMENT_TRACKER:
    print(f"[{row['status']:<5}] {row['id']} - {row['task']} (owner: {row['owner'] or 'unassigned'})")

[todo ] P0-1 - Confirmation write invariants (owner: unassigned)
[todo ] P0-2 - Company scope regression tests (owner: unassigned)
[todo ] P0-3 - BGN threshold test suite (owner: unassigned)
[todo ] P1-1 - Missing-fields response templates (owner: unassigned)
[todo ] P1-2 - Downstream error mapping (owner: unassigned)
[todo ] P2-1 - Tool latency + error observability (owner: unassigned)


## 4) Suggested Operating Loop

1. Pick one backlog item.
2. Implement code + tests.
3. Run this notebook's evaluation cases against new behavior.
4. Record score deltas and failure examples.
5. Keep changes only if safety/correctness is not regressed.

This gives you a tight improve-measure-improve loop for the accountant agent.

## 5) Live Debug Playground (Real-life Cases)

Edit inputs in the next cells and re-run to debug behavior.

You get:
- Live SSE event capture from `/agents/{agent_id}/chat`
- Token stream reconstruction
- Full conversation fetch from `/conversations/{conversation_id}`
- Pretty-printed timeline for fast inspection

In [98]:
import json
from datetime import datetime
from typing import Any

import httpx

# --- Edit these before running live tests ---
# This notebook talks directly to Agent service (internal service debugging).
# Gateway (:8000) is intentionally excluded to avoid auth/routing differences.
AGENT_BASE_URL_CANDIDATES = [
    "http://localhost:8002",  # direct agent service
]
AGENT_SERVICE_URL = AGENT_BASE_URL_CANDIDATES[0]
X_USER_ID = "69ee3344826015e6593e6c9e"
AGENT_ID = "69ee334437c7ac857d865c8d"
DEFAULT_TIMEOUT_SECONDS = 180.0

# Optional: reuse conversation across turns
ACTIVE_CONVERSATION_ID: str | None = None


def resolve_reachable_base_url(candidates: list[str]) -> str:
    for base in candidates:
        try:
            health_url = f"{base.rstrip('/')}/health"
            r = httpx.get(health_url, timeout=5.0)
            if r.status_code < 500:
                return base
        except Exception:
            continue
    raise RuntimeError(
        "No reachable Agent service base URL. Start docker services and verify agent on http://localhost:8002."
    )


AGENT_SERVICE_URL = resolve_reachable_base_url(AGENT_BASE_URL_CANDIDATES)
print("Set AGENT_ID before running chat tests.")
print(f"Resolved agent service URL: {AGENT_SERVICE_URL}")

Set AGENT_ID before running chat tests.
Resolved agent service URL: http://localhost:8002


In [99]:
# --- Agent discovery helper ---
# Use this cell to discover/set AGENT_ID for the current X_USER_ID,
# then validate the accountant tool surface expected by runtime/accountant.py.

PREFERRED_AGENT_TYPE = "accountant"
PREFERRED_AGENT_NAME_CONTAINS = ""  # optional substring filter, e.g. "main"
AUTO_SET_AGENT_ID = True


def list_user_agents(*, base_url: str | None = None, user_id: str | None = None) -> list[dict[str, Any]]:
    base = (base_url or AGENT_SERVICE_URL).rstrip("/")
    uid = user_id or X_USER_ID
    url = f"{base}/agents"
    headers = {"x-user-id": uid}
    try:
        r = httpx.get(url, headers=headers, timeout=DEFAULT_TIMEOUT_SECONDS)
        r.raise_for_status()
    except httpx.ConnectError as exc:
        raise RuntimeError(
            f"Cannot connect to {base}. Check docker services and agent port mapping (expected :8002)."
        ) from exc
    except httpx.HTTPStatusError as exc:
        status = exc.response.status_code
        raise RuntimeError(
            f"Agent service request failed with HTTP {status} at {url}. "
            "This notebook is configured for direct agent service access (no gateway)."
        ) from exc
    data = r.json()
    if not isinstance(data, list):
        raise TypeError(f"Expected list from /agents, got {type(data)!r}")
    return data


def get_agent(agent_id: str, *, base_url: str | None = None, user_id: str | None = None) -> dict[str, Any]:
    base = (base_url or AGENT_SERVICE_URL).rstrip("/")
    uid = user_id or X_USER_ID
    url = f"{base}/agents/{agent_id}"
    headers = {"x-user-id": uid}
    r = httpx.get(url, headers=headers, timeout=DEFAULT_TIMEOUT_SECONDS)
    r.raise_for_status()
    data = r.json()
    if not isinstance(data, dict):
        raise TypeError(f"Expected object from /agents/{{id}}, got {type(data)!r}")
    return data


def pick_agent(
    agents: list[dict[str, Any]],
    *,
    agent_type: str | None = PREFERRED_AGENT_TYPE,
    name_contains: str = PREFERRED_AGENT_NAME_CONTAINS,
) -> dict[str, Any] | None:
    filtered = agents
    if agent_type:
        filtered = [a for a in filtered if str(a.get("agent_type", "")).lower() == agent_type.lower()]
    if name_contains:
        needle = name_contains.lower().strip()
        filtered = [a for a in filtered if needle in str(a.get("name", "")).lower()]
    return filtered[0] if filtered else None


def expected_accountant_tool_surface() -> dict[str, list[str]]:
    expected_core_tools = ["rag_search", "calculator", "date_helper"]
    expected_builders = [
        "build_financial_tools",
        "build_partner_tools",
        "build_companybook_tools",
        "build_company_tools (only when company_id is None)",
    ]
    return {
        "core_tools": expected_core_tools,
        "tool_builders": expected_builders,
    }


def assert_accountant_tooling(agent: dict[str, Any]) -> None:
    agent_type = str(agent.get("agent_type") or "").lower()
    if agent_type != "accountant":
        raise AssertionError(
            f"Selected AGENT_ID={agent.get('id')} is type={agent_type!r}, expected 'accountant'."
        )

    surface = expected_accountant_tool_surface()
    print("\nExpected accountant tool surface:")
    print("- Core tools:")
    for tool_name in surface["core_tools"]:
        print(f"  - {tool_name}")
    print("- Tool builders:")
    for builder in surface["tool_builders"]:
        print(f"  - {builder}")


agents = list_user_agents()
print(f"Found {len(agents)} agents for X_USER_ID={X_USER_ID!r}\n")
for i, a in enumerate(agents, start=1):
    print(
        f"{i:02d}. id={a.get('id')} | name={a.get('name')} | "
        f"type={a.get('agent_type')} | company_id={a.get('company_id')}"
    )

picked = pick_agent(agents)
if picked:
    print("\nSelected agent:")
    print(
        json.dumps(
            {
                "id": picked.get("id"),
                "name": picked.get("name"),
                "agent_type": picked.get("agent_type"),
                "company_id": picked.get("company_id"),
            },
            indent=2,
            ensure_ascii=True,
        )
    )

    selected_id = str(picked["id"])
    if AUTO_SET_AGENT_ID:
        AGENT_ID = selected_id
        print(f"\nAGENT_ID auto-set to: {AGENT_ID}")

    selected_agent = get_agent(selected_id)
    assert_accountant_tooling(selected_agent)
else:
    print("\nNo matching agent found. Set AGENT_ID manually.")

Found 2 agents for X_USER_ID='69ee3344826015e6593e6c9e'

01. id=69f93225cb672064c4886a7b | name=baj pesho | type=inventory | company_id=69ee334437c7ac857d865c89
02. id=69ee334437c7ac857d865c8d | name=SNEJA | type=accountant | company_id=69ee334437c7ac857d865c89

Selected agent:
{
  "id": "69ee334437c7ac857d865c8d",
  "name": "SNEJA",
  "agent_type": "accountant",
  "company_id": "69ee334437c7ac857d865c89"
}

AGENT_ID auto-set to: 69ee334437c7ac857d865c8d

Expected accountant tool surface:
- Core tools:
  - rag_search
  - calculator
  - date_helper
- Tool builders:
  - build_financial_tools
  - build_partner_tools
  - build_companybook_tools
  - build_company_tools (only when company_id is None)


In [100]:
def parse_sse_lines(lines: list[str]) -> list[dict[str, Any]]:
    events: list[dict[str, Any]] = []
    current_event = "message"
    data_parts: list[str] = []

    def flush() -> None:
        nonlocal current_event, data_parts
        if not data_parts:
            return
        raw_data = "\n".join(data_parts)
        try:
            parsed_data = json.loads(raw_data)
        except json.JSONDecodeError:
            parsed_data = {"raw": raw_data}
        events.append({"event": current_event, "data": parsed_data})
        current_event = "message"
        data_parts = []

    for line in lines:
        if line == "":
            flush()
            continue
        if line.startswith(":"):
            continue
        if line.startswith("event:"):
            current_event = line.split(":", 1)[1].strip()
            continue
        if line.startswith("data:"):
            data_parts.append(line.split(":", 1)[1].strip())
            continue

    flush()
    return events


def stream_chat_debug(
    *,
    agent_id: str,
    message: str,
    conversation_id: str | None = None,
    base_url: str | None = None,
    user_id: str | None = None,
    timeout_seconds: float = DEFAULT_TIMEOUT_SECONDS,
    print_raw: bool = False,
) -> dict[str, Any]:
    if not agent_id:
        raise ValueError("AGENT_ID is empty. Set AGENT_ID first.")
    if not message.strip():
        raise ValueError("message is empty")

    base = (base_url or AGENT_SERVICE_URL).rstrip("/")
    uid = user_id or X_USER_ID
    url = f"{base}/agents/{agent_id}/chat"
    headers = {"x-user-id": uid, "accept": "text/event-stream"}
    payload: dict[str, Any] = {"message": message}
    if conversation_id:
        payload["conversation_id"] = conversation_id

    raw_lines: list[str] = []
    try:
        with httpx.stream(
            "POST",
            url,
            headers=headers,
            json=payload,
            timeout=timeout_seconds,
        ) as response:
            response.raise_for_status()
            for line in response.iter_lines():
                if print_raw:
                    print(line)
                raw_lines.append(line)
    except httpx.ConnectError as exc:
        raise RuntimeError(
            f"Cannot connect to {base}. Check docker services and agent port mapping (expected :8002)."
        ) from exc

    events = parse_sse_lines(raw_lines)
    assistant_text = "".join(
        e.get("data", {}).get("content", "")
        for e in events
        if e.get("event") == "token" and isinstance(e.get("data"), dict)
    )

    final_conversation_id = None
    for event in reversed(events):
        data = event.get("data")
        if isinstance(data, dict) and data.get("conversation_id"):
            final_conversation_id = str(data["conversation_id"])
            break

    return {
        "url": url,
        "payload": payload,
        "events": events,
        "assistant_text": assistant_text,
        "conversation_id": final_conversation_id,
        "raw_line_count": len(raw_lines),
    }


def fetch_conversation(
    conversation_id: str,
    *,
    base_url: str | None = None,
    user_id: str | None = None,
    timeout_seconds: float = DEFAULT_TIMEOUT_SECONDS,
) -> dict[str, Any]:
    base = (base_url or AGENT_SERVICE_URL).rstrip("/")
    uid = user_id or X_USER_ID
    url = f"{base}/conversations/{conversation_id}"
    headers = {"x-user-id": uid}
    r = httpx.get(url, headers=headers, timeout=timeout_seconds)
    r.raise_for_status()
    return r.json()

In [101]:
def pretty_print_events(events: list[dict[str, Any]], max_events: int = 80) -> None:
    clipped = events[:max_events]
    print(f"events: {len(events)} (showing {len(clipped)})")
    for i, e in enumerate(clipped, start=1):
        event_name = e.get("event", "?")
        data = e.get("data")
        if isinstance(data, dict):
            preview = json.dumps(data, ensure_ascii=True)
            if len(preview) > 220:
                preview = preview[:220] + " ..."
        else:
            preview = str(data)
        print(f"{i:02d}. {event_name:>12} | {preview}")


def _coerce_tool_call(raw_call: Any) -> dict[str, Any] | None:
    if not isinstance(raw_call, dict):
        return None
    function_obj = raw_call.get("function") if isinstance(raw_call.get("function"), dict) else {}
    raw_args = raw_call.get("args")
    if raw_args is None:
        raw_args = function_obj.get("arguments")

    args: Any = raw_args if raw_args is not None else {}
    if isinstance(args, str):
        try:
            args = json.loads(args)
        except Exception:
            pass

    return {
        "id": raw_call.get("id") or raw_call.get("tool_call_id"),
        "name": raw_call.get("name") or function_obj.get("name") or "unknown",
        "args": args,
    }


def _extract_tool_calls(msg: dict[str, Any]) -> list[dict[str, Any]]:
    containers: list[dict[str, Any]] = []
    if isinstance(msg, dict):
        containers.append(msg)
        for k in ("metadata", "additional_kwargs", "kwargs", "data"):
            v = msg.get(k)
            if isinstance(v, dict):
                containers.append(v)

    extracted: list[dict[str, Any]] = []
    for container in containers:
        tool_calls = container.get("tool_calls")
        if isinstance(tool_calls, list):
            for call in tool_calls:
                coerced = _coerce_tool_call(call)
                if coerced:
                    extracted.append(coerced)

        single_tool_call = container.get("tool_call")
        if isinstance(single_tool_call, dict):
            coerced = _coerce_tool_call(single_tool_call)
            if coerced:
                extracted.append(coerced)

    return extracted


def collect_tool_usage(events: list[dict[str, Any]], conversation: dict[str, Any] | None = None) -> list[dict[str, Any]]:
    usage: list[dict[str, Any]] = []

    for idx, event in enumerate(events, start=1):
        event_name = str(event.get("event", "")).lower()
        data = event.get("data") if isinstance(event.get("data"), dict) else {}

        for call in _extract_tool_calls(data):
            usage.append(
                {
                    "source": "event.tool_call",
                    "order": idx,
                    "name": call.get("name") or "unknown",
                    "call_id": call.get("id") or "n/a",
                }
            )

        if event_name in {"on_tool_start", "tool_start", "tool", "tool_call"}:
            tool_name = data.get("name") or data.get("tool") or data.get("tool_name") or "unknown"
            usage.append(
                {
                    "source": f"event.{event_name or 'tool'}",
                    "order": idx,
                    "name": str(tool_name),
                    "call_id": str(data.get("tool_call_id") or data.get("id") or "n/a"),
                }
            )

    if conversation:
        messages = conversation.get("messages", [])
        for idx, msg in enumerate(messages, start=1):
            role = str(msg.get("role", "")).lower()
            if role == "assistant":
                for call in _extract_tool_calls(msg):
                    usage.append(
                        {
                            "source": "conversation.assistant",
                            "order": idx,
                            "name": call.get("name") or "unknown",
                            "call_id": call.get("id") or "n/a",
                        }
                    )
            elif role == "tool":
                tool_name = msg.get("name") or (msg.get("metadata") or {}).get("name") or "unknown"
                usage.append(
                    {
                        "source": "conversation.tool_message",
                        "order": idx,
                        "name": str(tool_name),
                        "call_id": str(msg.get("tool_call_id") or "n/a"),
                    }
                )

    return usage


def print_tool_usage_report(events: list[dict[str, Any]], conversation: dict[str, Any] | None = None) -> None:
    usage = collect_tool_usage(events, conversation)
    print("Tool usage report")
    print("=" * 90)

    if not usage:
        print("No tool usage detected in SSE events or persisted conversation messages.")
        print("Tip: if the agent answers directly, no tools are invoked. You can force tool-heavy prompts to validate logging.")
        return

    counts: dict[str, int] = {}
    for row in usage:
        name = row["name"]
        counts[name] = counts.get(name, 0) + 1

    print(f"Total tool activity rows: {len(usage)}")
    print("By tool:")
    for name, count in sorted(counts.items(), key=lambda x: (-x[1], x[0])):
        print(f"- {name}: {count}")

    print("\nCall order:")
    for i, row in enumerate(usage, start=1):
        print(f"{i:02d}. {row['name']} | source={row['source']} | call_id={row['call_id']}")


def _print_message_like_trace(msg: dict[str, Any], max_content_chars: int = 1200) -> None:
    role = str(msg.get("role", "unknown")).lower()
    role_label_map = {
        "user": "Human Message",
        "assistant": "Ai Message",
        "tool": "Tool Message",
        "system": "System Message",
    }
    role_label = role_label_map.get(role, f"{role.title()} Message")

    print(f"\n{'=' * 30} {role_label} {'=' * 30}")

    if role == "assistant":
        tool_calls = _extract_tool_calls(msg)
        if tool_calls:
            print("Tool Calls:")
            for call in tool_calls:
                call_id = call.get("id") or call.get("tool_call_id") or "n/a"
                raw_name = call.get("name")
                function_obj = call.get("function") if isinstance(call.get("function"), dict) else {}
                name = raw_name or function_obj.get("name") or "unknown"

                raw_args = call.get("args")
                if raw_args is None:
                    raw_args = function_obj.get("arguments", {})
                if isinstance(raw_args, str):
                    try:
                        args = json.loads(raw_args)
                    except Exception:
                        args = raw_args
                else:
                    args = raw_args if raw_args is not None else {}

                print(f"  {name} ({call_id})")
                print(f"  Call ID: {call_id}")
                print("  Args:")
                if isinstance(args, dict):
                    for k, v in args.items():
                        print(f"    {k}: {v}")
                else:
                    print(f"    {args}")

    if role == "tool":
        tool_name = msg.get("name") or (msg.get("metadata") or {}).get("name") or "unknown"
        print(f"Name: {tool_name}")

    content = str(msg.get("content", "") or "")
    if len(content) > max_content_chars:
        content = content[:max_content_chars] + "\n...[truncated]"
    if content:
        print(content)


def pretty_print_conversation(conversation: dict[str, Any], max_content_chars: int = 1200) -> None:
    convo_id = conversation.get("id")
    title = conversation.get("title")
    agent_id = conversation.get("agent_id")
    created_at = conversation.get("created_at")
    updated_at = conversation.get("updated_at")
    messages = conversation.get("messages", [])

    print("=" * 90)
    print(f"Conversation: {convo_id}")
    print(f"Title      : {title}")
    print(f"Agent ID   : {agent_id}")
    print(f"Created At : {created_at}")
    print(f"Updated At : {updated_at}")
    print(f"Messages   : {len(messages)}")
    print("=" * 90)

    for msg in messages:
        _print_message_like_trace(msg, max_content_chars=max_content_chars)


def pretty_print_last_turn(conversation: dict[str, Any], max_content_chars: int = 1200) -> None:
    messages = conversation.get("messages", [])
    if not messages:
        pretty_print_conversation(conversation, max_content_chars=max_content_chars)
        return

    last_user_idx = None
    for i in range(len(messages) - 1, -1, -1):
        if str(messages[i].get("role", "")).lower() == "user":
            last_user_idx = i
            break

    if last_user_idx is None:
        pretty_print_conversation(conversation, max_content_chars=max_content_chars)
        return

    print("Last turn")
    print("=" * 90)
    for msg in messages[last_user_idx:]:
        _print_message_like_trace(msg, max_content_chars=max_content_chars)

In [102]:
# --- Interactive input cell ---
INPUT_MESSAGE = "Create an invoice draft to Acme for 1200 EUR consulting for April."
REUSE_ACTIVE_CONVERSATION = True
PRINT_RAW_SSE_LINES = False
PRINT_EVENT_TABLE = True
PRINT_TOOL_USAGE_REPORT = True
FETCH_AND_PRINT_FULL_CONVERSATION = True
PRINT_ONLY_LAST_TURN = True

conversation_for_request = ACTIVE_CONVERSATION_ID if REUSE_ACTIVE_CONVERSATION else None

result = stream_chat_debug(
    agent_id=AGENT_ID,
    message=INPUT_MESSAGE,
    conversation_id=conversation_for_request,
    print_raw=PRINT_RAW_SSE_LINES,
)

print(f"Request URL        : {result['url']}")
print(f"Conversation ID    : {result['conversation_id']}")
print(f"Raw SSE line count : {result['raw_line_count']}")
print(f"Assistant chars    : {len(result['assistant_text'])}")

if PRINT_EVENT_TABLE:
    pretty_print_events(result["events"])

if result["conversation_id"]:
    ACTIVE_CONVERSATION_ID = result["conversation_id"]

conversation_data: dict[str, Any] | None = None
if FETCH_AND_PRINT_FULL_CONVERSATION and result["conversation_id"]:
    conversation_data = fetch_conversation(result["conversation_id"])
    if PRINT_ONLY_LAST_TURN:
        pretty_print_last_turn(conversation_data)
    else:
        pretty_print_conversation(conversation_data)
else:
    print("No conversation_id found in SSE events.")

if PRINT_TOOL_USAGE_REPORT:
    print()
    print_tool_usage_report(result["events"], conversation_data)

Request URL        : http://localhost:8002/agents/69ee334437c7ac857d865c8d/chat
Conversation ID    : 69f9cbbbf944e03ed3a2260d
Raw SSE line count : 222
Assistant chars    : 271
events: 74 (showing 74)
01. conversation | {"conversation_id": "69f9cbbbf944e03ed3a2260d"}
02.        start | {"conversation_id": "69f9cbbbf944e03ed3a2260d"}
03.        token | {"content": "I"}
04.        token | {"content": " have"}
05.        token | {"content": " prepared"}
06.        token | {"content": " a"}
07.        token | {"content": " draft"}
08.        token | {"content": " invoice"}
09.        token | {"content": " for"}
10.        token | {"content": " Ac"}
11.        token | {"content": "me"}
12.        token | {"content": " Client"}
13.        token | {"content": " Ltd"}
14.        token | {"content": "."}
15.        token | {"content": " for"}
16.        token | {"content": " consulting"}
17.        token | {"content": " services"}
18.        token | {"content": " for"}
19.        token | {"conte

In [103]:
REAL_LIFE_DEBUG_CASES = [
    {
        "name": "Tax explanation with one retrieval",
        "message": "For my company, explain VAT treatment for this quarter and cite what you used.",
        "why": "Checks RAG usage discipline + grounded response.",
    },
    {
        "name": "Invoice requires draft+confirm",
        "message": "Create an invoice to Contoso for 1500 EUR software services.",
        "why": "Checks no immediate write without explicit confirmation.",
    },
    {
        "name": "Expense confirmation gate",
        "message": "Record expense: office supplies 320 BGN, paid today.",
        "why": "Checks confirmation before persistence.",
    },
    {
        "name": "Currency threshold logic (RAG + financial)",
        "message": "Use rag_search once to retrieve the current Bulgarian VAT registration threshold source, then check whether we are above threshold based on our EUR sales totals.",
        "why": "Checks grounded threshold retrieval + source currency vs BGN conversion logic.",
    },
    {
        "name": "Missing fields behavior",
        "message": "Create invoice for Beta Ltd.",
        "why": "Checks focused questions for missing required fields only.",
    },
    {
        "name": "Latest invoice in Nov 2025",
        "message": "Get my latest invoice from November 2025.",
        "why": "Checks invoice retrieval with month filter and correct latest-by-date selection.",
    },
]

for i, c in enumerate(REAL_LIFE_DEBUG_CASES, start=1):
    print(f"{i}. {c['name']}\n   message: {c['message']}\n   check  : {c['why']}\n")

1. Tax explanation with one retrieval
   message: For my company, explain VAT treatment for this quarter and cite what you used.
   check  : Checks RAG usage discipline + grounded response.

2. Invoice requires draft+confirm
   message: Create an invoice to Contoso for 1500 EUR software services.
   check  : Checks no immediate write without explicit confirmation.

3. Expense confirmation gate
   message: Record expense: office supplies 320 BGN, paid today.
   check  : Checks confirmation before persistence.

4. Currency threshold logic (RAG + financial)
   message: Use rag_search once to retrieve the current Bulgarian VAT registration threshold source, then check whether we are above threshold based on our EUR sales totals.
   check  : Checks grounded threshold retrieval + source currency vs BGN conversion logic.

5. Missing fields behavior
   message: Create invoice for Beta Ltd.
   check  : Checks focused questions for missing required fields only.

6. Latest invoice in Nov 2025
   

In [104]:
import re


_TOOL_TAG_PATTERN = re.compile(r"\[(tool\.start|tool\.end\s*|tool\.error)\]")
_CTX_PATTERN = re.compile(r"agent=(?P<agent_type>[\w-]+)/(?P<agent_id>[\w-]+)\s+user=(?P<user_id>[\w-]+)\s+run=(?P<run_id>[\w-]+)")
_ANSI_PATTERN = re.compile(r"\x1B\[[0-?]*[ -/]*[@-~]")


def _strip_compose_prefix(line: str) -> str:
    # docker compose logs often prefix lines with "service-1  |"
    clean = _ANSI_PATTERN.sub("", line)
    if "|" in clean:
        return clean.split("|", 1)[1].strip()
    return clean.strip()


def _collect_service_logs(*, service_names: list[str], since: str) -> tuple[str, list[str]]:
    compose_base = str(ROOT / "docker-compose.yml")
    compose_dev = str(ROOT / "docker-compose.dev.yml")

    collected_chunks: list[str] = []
    diagnostics: list[str] = []

    for service_name in service_names:
        cmd = [
            "docker",
            "compose",
            "-f",
            compose_base,
            "-f",
            compose_dev,
            "logs",
            service_name,
            "--since",
            since,
            "--no-color",
        ]
        try:
            proc = subprocess.run(cmd, capture_output=True, text=True, check=False, cwd=str(ROOT))
        except Exception as exc:
            diagnostics.append(f"compose logs failed for {service_name}: {exc}")
            continue

        merged = ((proc.stdout or "") + "\n" + (proc.stderr or "")).strip()
        if not merged:
            diagnostics.append(f"compose logs empty for {service_name} (code={proc.returncode})")
            continue

        if proc.returncode != 0:
            diagnostics.append(f"compose logs non-zero for {service_name} (code={proc.returncode})")
            continue

        collected_chunks.append(merged)

    # Fallback: collect from all services in project and parse only matching tags.
    if not collected_chunks:
        cmd_all = [
            "docker",
            "compose",
            "-f",
            compose_base,
            "-f",
            compose_dev,
            "logs",
            "--since",
            since,
            "--no-color",
        ]
        try:
            proc_all = subprocess.run(cmd_all, capture_output=True, text=True, check=False, cwd=str(ROOT))
            merged_all = ((proc_all.stdout or "") + "\n" + (proc_all.stderr or "")).strip()
            if proc_all.returncode == 0 and merged_all:
                collected_chunks.append(merged_all)
            else:
                diagnostics.append(f"compose logs (all services) failed (code={proc_all.returncode})")
        except Exception as exc:
            diagnostics.append(f"compose logs (all services) exception: {exc}")

    return "\n".join(collected_chunks).strip(), diagnostics


def parse_agent_runtime_tool_events(raw_logs: str) -> list[dict[str, Any]]:
    if not raw_logs.strip():
        return []

    normalized = [_strip_compose_prefix(line) for line in raw_logs.splitlines() if line.strip()]
    events: list[dict[str, Any]] = []
    current: dict[str, Any] | None = None

    def flush_current() -> None:
        nonlocal current
        if current is not None:
            events.append(current)
            current = None

    for line in normalized:
        tag_match = _TOOL_TAG_PATTERN.search(line)
        if tag_match:
            flush_current()
            tag = tag_match.group(1).replace(" ", "")
            status = "start" if tag == "tool.start" else "end" if tag.startswith("tool.end") else "error"
            tool_name = line.split("]", 1)[1].strip() if "]" in line else "unknown"
            current = {
                "status": status,
                "tool": tool_name,
                "agent_type": None,
                "agent_id": None,
                "user_id": None,
                "run_id": None,
                "input": None,
                "output": None,
                "error": None,
                "raw": [line],
            }
            continue

        if current is None:
            continue

        current["raw"].append(line)
        ctx = _CTX_PATTERN.search(line)
        if ctx:
            current["agent_type"] = ctx.group("agent_type")
            current["agent_id"] = ctx.group("agent_id")
            current["user_id"] = ctx.group("user_id")
            current["run_id"] = ctx.group("run_id")
            continue

        if line.startswith("input:"):
            current["input"] = line.split(":", 1)[1].strip()
        elif line.startswith("output:"):
            current["output"] = line.split(":", 1)[1].strip()
        elif line.startswith("error:"):
            current["error"] = line.split(":", 1)[1].strip()

    flush_current()
    return events


def read_agent_runtime_tool_events(
    *,
    conversation_id: str | None,
    since: str = "15m",
    service_names: list[str] | None = None,
    max_events: int = 120,
) -> list[dict[str, Any]]:
    services = service_names or ["agent", "agent-service", "agent_app"]
    raw_logs, diagnostics = _collect_service_logs(service_names=services, since=since)

    if diagnostics:
        print("Log collection diagnostics:")
        for line in diagnostics[:8]:
            print(f"- {line}")

    events = parse_agent_runtime_tool_events(raw_logs)

    agent_id = globals().get("AGENT_ID")
    if agent_id:
        filtered = [e for e in events if str(e.get("agent_id") or "") == str(agent_id)]
        if filtered:
            events = filtered

    # conversation_id is not directly present in these log lines; we keep the
    # parameter for API compatibility and future correlation if IDs are added.
    _ = conversation_id

    return events[-max_events:]


def print_agent_runtime_tool_logs(
    *,
    conversation_id: str | None,
    since: str = "15m",
    service_name: str = "agent",
    max_lines: int = 80,
) -> None:
    events = read_agent_runtime_tool_events(
        conversation_id=conversation_id,
        since=since,
        service_names=[service_name, "agent-service", "agent_app"],
        max_events=max_lines,
    )

    print("\nBackend runtime tool logs (structured):")
    print("-" * 100)

    if not events:
        print("No tool events found.")
        print("Tips:")
        print("- Increase since='15m' or '30m'.")
        print("- Run a tool-heavy case right before reading logs.")
        print("- Ensure agent service logs are in development mode.")
        print("-" * 100)
        return

    counts: dict[str, int] = {}
    for e in events:
        key = str(e.get("tool") or "unknown")
        counts[key] = counts.get(key, 0) + 1

    print("Tool totals:")
    for name, count in sorted(counts.items(), key=lambda x: (-x[1], x[0])):
        print(f"- {name}: {count}")

    print("\nTimeline:")
    for i, e in enumerate(events, start=1):
        status = str(e.get("status") or "?")
        tool = str(e.get("tool") or "unknown")
        run_id = str(e.get("run_id") or "n/a")
        payload = e.get("input") if status == "start" else e.get("output") if status == "end" else e.get("error")
        payload_preview = str(payload or "")
        if len(payload_preview) > 180:
            payload_preview = payload_preview[:180] + " ..."
        print(f"{i:02d}. [{status:5}] {tool} | run={run_id} | {payload_preview}")

    print("-" * 100)


In [105]:
def run_case(
    case: dict[str, str],
    *,
    reuse_conversation: bool = True,
    print_last_turn: bool = True,
    print_backend_tool_logs: bool = True,
    backend_logs_since: str = "15m",
) -> dict[str, Any]:
    global ACTIVE_CONVERSATION_ID

    convo_id = ACTIVE_CONVERSATION_ID if reuse_conversation else None
    out = stream_chat_debug(
        agent_id=AGENT_ID,
        message=case["message"],
        conversation_id=convo_id,
        print_raw=False,
    )

    conversation_id = out.get("conversation_id")
    if conversation_id:
        ACTIVE_CONVERSATION_ID = conversation_id

    print(f"Case: {case['name']}")
    print(f"Why : {case['why']}")
    print(f"Conversation ID: {conversation_id}")
    print("Assistant preview:")
    print("-" * 60)
    print((out.get("assistant_text") or "")[:1200])
    print("-" * 60)

    if print_last_turn and conversation_id:
        conversation = fetch_conversation(conversation_id)
        pretty_print_last_turn(conversation)

        usage = collect_tool_usage(out.get("events", []), conversation)
        if usage:
            print("\nTool Calls (detected):")
            seen: set[tuple[str, str]] = set()
            for row in usage:
                name = str(row.get("name") or "unknown")
                call_id = str(row.get("call_id") or "n/a")
                key = (name, call_id)
                if key in seen:
                    continue
                seen.add(key)
                print(f"- {name} ({call_id})")
        else:
            print("\nTool Calls: none detected in SSE events/conversation for this turn.")

    if print_backend_tool_logs:
        print_agent_runtime_tool_logs(
            conversation_id=conversation_id,
            since=backend_logs_since,
        )

    return out

# Example usage:
case_result = run_case(
    REAL_LIFE_DEBUG_CASES[5],
    reuse_conversation=True,
    print_last_turn=True,
    print_backend_tool_logs=True,
    backend_logs_since="10m",
)

Case: Latest invoice in Nov 2025
Why : Checks invoice retrieval with month filter and correct latest-by-date selection.
Conversation ID: 69f9cbbbf944e03ed3a2260d
Assistant preview:
------------------------------------------------------------
Your latest invoice from November 2025 is to the client "РОБО ЛАБ" for payroll processing. The invoice number is INV-2025-00003, issued on 2025-11-24, with a total amount of 1211.12 EUR including 20% VAT. The invoice status is paid.

If you need any more details or actions related to this invoice, please let me know.
------------------------------------------------------------
Last turn

============================== Human Message ==============================
Get my latest invoice from November 2025.

============================== Ai Message ==============================
Your latest invoice from November 2025 is to the client "РОБО ЛАБ" for payroll processing. The invoice number is INV-2025-00003, issued on 2025-11-24, with a total amount of 1